# Olist E-Commerce — Exploratory Data Analysis

This notebook explores the raw Olist dataset to understand structure, quality, and key patterns before building the transformation pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
RAW_DIR = Path("../data/raw")

## 1. Load All Datasets

In [ ]:
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
geo = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
translations = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

datasets = {
    "orders": orders, "items": items, "payments": payments,
    "reviews": reviews, "products": products, "customers": customers,
    "sellers": sellers, "geolocation": geo, "translations": translations
}

for name, df in datasets.items():
    print(f"{name:>15}: {len(df):>10,} rows  x  {len(df.columns)} cols")

## 2. Data Quality Assessment

In [ ]:
for name, df in datasets.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"\n--- {name} ---")
        for col, count in nulls.items():
            pct = 100 * count / len(df)
            print(f"  {col}: {count:,} nulls ({pct:.1f}%)")

## 3. Order Status Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
orders["order_status"].value_counts().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Order Status Distribution")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

## 4. Monthly Order Volume

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
monthly = orders.set_index("order_purchase_timestamp").resample("ME")["order_id"].count()

fig, ax = plt.subplots(figsize=(12, 5))
monthly.plot(ax=ax, marker="o", color="steelblue")
ax.set_title("Monthly Order Volume")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.show()

## 5. Review Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
reviews["review_score"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Review Score Distribution")
ax.set_xlabel("Score")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Payment Method Breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
payments["payment_type"].value_counts().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Payment Method Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 7. Top 10 Product Categories by Revenue

In [ ]:
merged = items.merge(products, on="product_id").merge(translations, on="product_category_name", how="left")
top_cats = merged.groupby("product_category_name_english")["price"].sum().nlargest(10)

fig, ax = plt.subplots(figsize=(10, 6))
top_cats.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 10 Categories by Revenue")
ax.set_xlabel("Revenue (R$)")
plt.tight_layout()
plt.show()

## 8. Geolocation — Customer State Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
customers["customer_state"].value_counts().head(15).plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Customers by State (Top 15)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 9. Delivery Time Distribution

In [ ]:
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
delivered = orders.dropna(subset=["order_delivered_customer_date"]).copy()
delivered["delivery_days"] = (delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]).dt.days

fig, ax = plt.subplots(figsize=(10, 5))
delivered["delivery_days"].clip(0, 60).hist(bins=60, ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Delivery Time Distribution (capped at 60 days)")
ax.set_xlabel("Days")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.show()

print(f"Median delivery: {delivered['delivery_days'].median():.0f} days")
print(f"95th percentile: {delivered['delivery_days'].quantile(0.95):.0f} days")

## Summary

Key findings from EDA:
- Dataset has 9 interrelated tables with clear join paths
- Significant nulls in delivery dates (undelivered orders) and review comments
- Geolocation has massive duplication per zip code (needs dedup)
- Date columns stored as strings (need parsing)
- Strong category concentration in a few product types
- Credit card is the dominant payment method
- Median delivery is ~12 days with a long tail

These findings directly inform the transformation and modeling design.